In [9]:
import pandas as pd
import glob
import os

# =========================
# Read Files
# =========================

sec_df = pd.read_csv(
    "/lakehouse/default/Files/data/processed/securities/security_processed.csv"
)

latest_nse_mcap = max(
    glob.glob(
        "/lakehouse/default/Files/data/raw/nse_mcap/*.csv"
    ),
    key=os.path.getmtime
)

nse_mcap = pd.read_csv(
    latest_nse_mcap
)

bse_mcap = pd.read_csv(
    "/lakehouse/default/Files/data/raw/bse_mcap/bse_mcap.csv"
)

sector_df = pd.read_csv(
    "/lakehouse/default/Files/data/processed/securities/sector_industry_lookup.csv"
)

# =========================
# Create Base Security Master
# =========================

sec_master = (
    sec_df
    .sort_values(
        ["isin", "source"]
    )
    .drop_duplicates(
        subset=["isin"],
        keep="first"
    )
    .copy()
)



# =========================
# NSE Market Cap
# =========================

nse_mcap_clean = nse_mcap[
    [
        "Symbol",
        "Close Price/Paid up value(Rs.)",
        "Market Cap(Rs.)              "
    ]
].copy()

nse_mcap_clean = nse_mcap_clean.rename(
    columns={
        "Symbol": "symbol",
        "Close Price/Paid up value(Rs.)": "cmp",
        "Market Cap(Rs.)              ": "nse_market_cap"
    }
)

# Convert NSE Market Cap from Rupees to Crores
nse_mcap_clean["nse_market_cap"] = (
    nse_mcap_clean["nse_market_cap"] / 10000000
)

sec_master = sec_master.merge(
    nse_mcap_clean,
    on="symbol",
    how="left"
)

# =========================
# BSE Market Cap
# =========================

bse_mcap_clean = bse_mcap[
    [
        "ISIN_NUMBER",
        "Mktcap"
    ]
].copy()

bse_mcap_clean = bse_mcap_clean.rename(
    columns={
        "ISIN_NUMBER": "isin",
        "Mktcap": "bse_market_cap"
    }
)

bse_mcap_clean = (
    bse_mcap_clean
    .dropna(
        subset=["isin"]
    )
)

sec_master = sec_master.merge(
    bse_mcap_clean,
    on="isin",
    how="left"
)

# =========================
# Final Market Cap
# =========================

sec_master["market_cap"] = (
    sec_master["nse_market_cap"]
    .fillna(
        sec_master["bse_market_cap"]
    )
)

# =========================
# Sector / Industry
# =========================

sector_df = sector_df[
    [
        "symbol",
        "sector",
        "industry"
    ]
].copy()

sector_df = sector_df.drop_duplicates(
    subset=["symbol"]
)

sec_master = sec_master.merge(
    sector_df,
    on="symbol",
    how="left"
)

# =========================
# Final Output
# =========================

security_master = sec_master[
    [
        "isin",
        "symbol",
        "company_name",
        "bse_security_code",
        "cmp",
        "market_cap",
        "sector",
        "industry",
        "face_value"
    ]
].copy()

print("SUCCESS")
print("Rows:", len(security_master))

print(
    "CMP Coverage:",
    security_master["cmp"].notna().sum()
)

print(
    "Market Cap Coverage:",
    security_master["market_cap"].notna().sum()
)

print(
    "Sector Coverage:",
    security_master["sector"].notna().sum()
)

print(
    "Industry Coverage:",
    security_master["industry"].notna().sum()
)

StatementMeta(, 50b03fee-38a4-4a65-9f6c-13500c02e54f, 11, Finished, Available, Finished, False)

SUCCESS
Rows: 5209
CMP Coverage: 2377
Market Cap Coverage: 4907
Sector Coverage: 1497
Industry Coverage: 1497


In [10]:
security_master_spark = spark.createDataFrame(
    security_master
)

spark.sql(
    "DROP TABLE IF EXISTS sec_master"
)

(
    security_master_spark
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(
        "sec_master"
    )
)

print("SUCCESS")

print(
    "Rows:",
    security_master_spark.count()
)

security_master_spark.printSchema()

display(
    security_master_spark.limit(10)
)

StatementMeta(, 50b03fee-38a4-4a65-9f6c-13500c02e54f, 12, Finished, Available, Finished, False)

SUCCESS
Rows: 5209
root
 |-- isin: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- bse_security_code: double (nullable = true)
 |-- cmp: double (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- sector: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- face_value: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 73996b56-3c19-4ca5-a63b-e23ba105eda3)

In [3]:
# from pyspark.sql import functions as F

# sec = spark.table("sec_master")

# print("Rows:", sec.count())

# print(
#     "CMP Coverage:",
#     sec.filter(
#         F.col("cmp").isNotNull()
#     ).count()
# )

# print(
#     "Market Cap Coverage:",
#     sec.filter(
#         F.col("market_cap").isNotNull()
#     ).count()
# )

# print(
#     "Sector Coverage:",
#     sec.filter(
#         F.col("sector").isNotNull()
#     ).count()
# )

# print(
#     "Industry Coverage:",
#     sec.filter(
#         F.col("industry").isNotNull()
#     ).count()
# )

StatementMeta(, c143a414-b717-4e6f-9112-edc69cb3248e, 5, Finished, Available, Finished, False)